<a href="https://colab.research.google.com/github/mmbc560/GUIA2/blob/main/Activity_%E2%80%93_Monte_Carlo_Supply_Chain_Challenge.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ==================================================================================================
# MONTE CARLO SIMULATION FOR INVENTORY, SOURCING AND SUPPLY CHAIN RISK
# ==================================================================================================
#
# OBJETIVO GENERAL
# --------------------------------------------------------------------------------------------------
# Simular 1.000 escenarios posibles para una cadena de suministro incorporando incertidumbre en:
#
#   1. DEMAND                → Demanda
#   2. LEAD TIME             → Tiempo de entrega del proveedor
#   3. SUPPLIER RELIABILITY  → Confiabilidad del proveedor
#
# Para cada escenario queremos calcular:
#
#   - Final Inventory
#   - Stockout
#   - Stockout Units
#   - Service Level
#   - Inventory Holding Cost
#   - Stockout Cost
#   - Emergency Cost
#   - Total Cost
#
# Luego compararemos diferentes estrategias:
#
#   A. Current Policy
#   B. Higher Safety Stock
#   C. Dual Sourcing
#   D. Nearshoring
#
#
# PREGUNTA GERENCIAL PRINCIPAL
# --------------------------------------------------------------------------------------------------
#
# ¿Qué estrategia ofrece el mejor equilibrio entre:
#
#         COST + SERVICE + RISK?
#
#
# IMPORTANTE:
# --------------------------------------------------------------------------------------------------
# Este modelo tiene fines académicos.
#
# Los parámetros y fórmulas pueden adaptarse posteriormente
# con datos reales de una organización.
#
# ==================================================================================================


# ==================================================================================================
# 1. IMPORTAR LIBRERÍAS
# ==================================================================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# ==================================================================================================
# 2. SEMILLA ALEATORIA
# ==================================================================================================
#
# Monte Carlo utiliza números aleatorios.
#
# np.random.seed() permite que los resultados puedan reproducirse.
#
# Esto significa que si los estudiantes ejecutan nuevamente el código
# utilizando la misma semilla, obtendrán exactamente los mismos números.
#
# Esto es muy útil en clase porque todos pueden analizar los mismos resultados.
#
# ==================================================================================================

np.random.seed(42)


# ==================================================================================================
# 3. NÚMERO DE SIMULACIONES
# ==================================================================================================
#
# Realizaremos 1.000 escenarios posibles.
#
# Cada escenario puede interpretarse como un posible futuro
# de la cadena de suministro.
#
# ==================================================================================================

N_SIMULATIONS = 1000


# ==================================================================================================
# 4. SUPUESTOS GENERALES DEL NEGOCIO
# ==================================================================================================
#
# Estos parámetros serán iguales para todas las estrategias.
#
# Después los estudiantes pueden modificarlos para crear nuevas situaciones.
#
# ==================================================================================================


# Demanda promedio esperada durante el periodo de análisis.
AVERAGE_DEMAND = 10000


# Desviación estándar de la demanda.
#
# Mientras mayor sea este valor,
# mayor será la incertidumbre de la demanda.
DEMAND_STD = 1500


# Precio o valor económico por unidad.
UNIT_VALUE = 40


# Costo anualizado de mantenimiento de inventario.
#
# 22% significa que mantener $100 en inventario
# cuesta aproximadamente $22 en el periodo considerado.
HOLDING_RATE = 0.22


# Costo por unidad no atendida.
#
# Puede representar:
# - ventas perdidas,
# - penalizaciones,
# - pérdida de margen,
# - costos de servicio.
STOCKOUT_COST_PER_UNIT = 25


# Costo adicional por una reposición de emergencia.
EMERGENCY_COST_PER_UNIT = 12


# ==================================================================================================
# 5. DEFINICIÓN DE ESTRATEGIAS
# ==================================================================================================
#
# Cada estrategia tendrá:
#
# - Initial Inventory
# - Safety Stock
# - Lead Time promedio
# - Variabilidad del Lead Time
# - Supplier Reliability promedio
# - Costo adicional de implementación
#
#
# La intención es mostrar que las estrategias tienen trade-offs.
#
# Por ejemplo:
#
# MAYOR SAFETY STOCK
#   + reduce faltantes
#   - aumenta capital de trabajo
#
# DUAL SOURCING
#   + reduce dependencia
#   + mejora continuidad
#   - puede aumentar costos de coordinación
#
# NEARSHORING
#   + reduce lead time
#   + aumenta capacidad de respuesta
#   - normalmente puede tener mayor costo de compra
#
# ==================================================================================================


strategies = {

    "Current Policy": {

        "initial_inventory": 8000,

        "safety_stock": 1500,

        "lead_time_mean": 25,

        "lead_time_std": 6,

        "reliability_mean": 0.86,

        "reliability_std": 0.07,

        "strategy_fixed_cost": 0
    },


    "Higher Safety Stock": {

        "initial_inventory": 8000,

        "safety_stock": 3000,

        "lead_time_mean": 25,

        "lead_time_std": 6,

        "reliability_mean": 0.86,

        "reliability_std": 0.07,

        "strategy_fixed_cost": 15000
    },


    "Dual Sourcing": {

        "initial_inventory": 8000,

        "safety_stock": 2000,

        "lead_time_mean": 19,

        "lead_time_std": 4,

        "reliability_mean": 0.94,

        "reliability_std": 0.04,

        "strategy_fixed_cost": 30000
    },


    "Nearshoring": {

        "initial_inventory": 8000,

        "safety_stock": 1800,

        "lead_time_mean": 12,

        "lead_time_std": 3,

        "reliability_mean": 0.96,

        "reliability_std": 0.03,

        "strategy_fixed_cost": 45000
    }

}


# ==================================================================================================
# 6. FUNCIÓN DE SIMULACIÓN MONTE CARLO
# ==================================================================================================
#
# Vamos a crear una función para simular cualquier estrategia.
#
# La función realizará N_SIMULATIONS escenarios.
#
#
# EN CADA ESCENARIO:
#
#   Paso 1 → Generamos una demanda aleatoria
#   Paso 2 → Generamos un lead time aleatorio
#   Paso 3 → Generamos confiabilidad del proveedor
#   Paso 4 → Calculamos cuánto suministro realmente llega
#   Paso 5 → Calculamos inventario disponible
#   Paso 6 → Calculamos stockout
#   Paso 7 → Calculamos service level
#   Paso 8 → Calculamos costos
#
# ==================================================================================================


def monte_carlo_strategy(strategy_name, parameters):


    # ----------------------------------------------------------------------------------------------
    # Creamos una lista vacía.
    #
    # Aquí almacenaremos los resultados
    # de cada una de las 1.000 simulaciones.
    # ----------------------------------------------------------------------------------------------

    results = []


    # ----------------------------------------------------------------------------------------------
    # Iniciamos Monte Carlo.
    # ----------------------------------------------------------------------------------------------

    for simulation in range(N_SIMULATIONS):


        # ==========================================================================================
        # STEP 1. SIMULATE DEMAND
        # ==========================================================================================
        #
        # Utilizamos una distribución normal.
        #
        # Ejemplo:
        #
        # promedio = 10.000 unidades
        # desviación = 1.500 unidades
        #
        # En algunas simulaciones la demanda puede ser:
        #
        # 8.500
        # 9.700
        # 10.500
        # 12.000
        #
        # etc.
        #
        # ==========================================================================================

        demand = np.random.normal(

            AVERAGE_DEMAND,

            DEMAND_STD

        )


        # Evitamos valores negativos.
        demand = max(
            demand,
            0
        )


        # ==========================================================================================
        # STEP 2. SIMULATE SUPPLIER LEAD TIME
        # ==========================================================================================
        #
        # El lead time tampoco es constante.
        #
        # Por ejemplo:
        #
        # promedio = 25 días
        #
        # pero puede haber entregas en:
        #
        # 18 días
        # 23 días
        # 31 días
        # 38 días
        #
        # ==========================================================================================

        lead_time = np.random.normal(

            parameters["lead_time_mean"],

            parameters["lead_time_std"]

        )


        # Evitamos valores irreales negativos.
        lead_time = max(
            lead_time,
            1
        )


        # ==========================================================================================
        # STEP 3. SIMULATE SUPPLIER RELIABILITY
        # ==========================================================================================
        #
        # Reliability representa qué porcentaje del pedido
        # logra llegar correctamente.
        #
        # Ejemplo:
        #
        # reliability = 0.90
        #
        # significa aproximadamente:
        #
        # 90% del suministro esperado llega correctamente.
        #
        # ==========================================================================================

        reliability = np.random.normal(

            parameters["reliability_mean"],

            parameters["reliability_std"]

        )


        # Reliability debe estar entre 0 y 1.
        reliability = np.clip(

            reliability,

            0,

            1

        )


        # ==========================================================================================
        # STEP 4. DETERMINE PLANNED REPLENISHMENT
        # ==========================================================================================
        #
        # Supongamos que la empresa intenta reponer
        # aproximadamente la demanda promedio.
        #
        # ==========================================================================================

        planned_replenishment = AVERAGE_DEMAND


        # ==========================================================================================
        # STEP 5. EFFECT OF LEAD TIME
        # ==========================================================================================
        #
        # A mayor lead time,
        # mayor exposición al riesgo.
        #
        # Creamos un factor simple:
        #
        # Lead time normal de referencia = 20 días.
        #
        # Si el proveedor tarda más,
        # aumenta el consumo antes de recibir el pedido.
        #
        # ==========================================================================================

        lead_time_factor = lead_time / 20


        # ==========================================================================================
        # STEP 6. ACTUAL SUPPLY RECEIVED
        # ==========================================================================================
        #
        # No todo el abastecimiento esperado necesariamente llega.
        #
        # Multiplicamos:
        #
        # pedido planeado × confiabilidad
        #
        # ==========================================================================================

        actual_supply = (

            planned_replenishment
            *
            reliability

        )


        # ==========================================================================================
        # STEP 7. EFFECTIVE DEMAND DURING SUPPLY EXPOSURE
        # ==========================================================================================
        #
        # Cuando el lead time aumenta,
        # la cadena queda expuesta durante más tiempo.
        #
        # Utilizamos un ajuste didáctico.
        #
        # ==========================================================================================

        adjusted_demand = (

            demand
            *
            (0.70 + 0.30 * lead_time_factor)

        )


        # ==========================================================================================
        # STEP 8. TOTAL INVENTORY AVAILABLE
        # ==========================================================================================
        #
        # El inventario disponible será:
        #
        # Initial Inventory
        # +
        # Safety Stock
        # +
        # Supply Received
        #
        # ==========================================================================================

        inventory_available = (

            parameters["initial_inventory"]

            +

            parameters["safety_stock"]

            +

            actual_supply

        )


        # ==========================================================================================
        # STEP 9. FINAL INVENTORY
        # ==========================================================================================
        #
        # Inventario final:
        #
        # disponibilidad - demanda
        #
        # ==========================================================================================

        final_inventory = (

            inventory_available

            -

            adjusted_demand

        )


        # ==========================================================================================
        # STEP 10. STOCKOUT
        # ==========================================================================================
        #
        # Si el inventario final es negativo,
        # significa que tuvimos faltantes.
        #
        # ==========================================================================================

        stockout_units = max(

            -final_inventory,

            0

        )


        # Creamos una variable binaria.
        #
        # 1 = ocurrió stockout
        # 0 = no ocurrió

        stockout = (

            1
            if stockout_units > 0
            else 0

        )


        # ==========================================================================================
        # STEP 11. INVENTORY CANNOT BE NEGATIVE
        # ==========================================================================================

        ending_inventory = max(

            final_inventory,

            0

        )


        # ==========================================================================================
        # STEP 12. SERVICE LEVEL
        # ==========================================================================================
        #
        # El nivel de servicio puede calcularse de diferentes maneras.
        #
        # Aquí utilizaremos:
        #
        # Service Level =
        #
        # 1 - Stockout Units / Demand
        #
        # ==========================================================================================

        if adjusted_demand > 0:

            service_level = (

                1

                -

                stockout_units
                /
                adjusted_demand

            ) * 100

        else:

            service_level = 100


        # Limitamos entre 0 y 100.
        service_level = np.clip(

            service_level,

            0,

            100

        )


        # ==========================================================================================
        # STEP 13. INVENTORY HOLDING COST
        # ==========================================================================================
        #
        # Cuanto mayor sea el inventario restante,
        # mayor es el costo de mantenerlo.
        #
        # ==========================================================================================

        inventory_holding_cost = (

            ending_inventory

            *
            UNIT_VALUE

            *
            HOLDING_RATE

        )


        # ==========================================================================================
        # STEP 14. STOCKOUT COST
        # ==========================================================================================

        stockout_cost = (

            stockout_units

            *
            STOCKOUT_COST_PER_UNIT

        )


        # ==========================================================================================
        # STEP 15. EMERGENCY REPLENISHMENT COST
        # ==========================================================================================
        #
        # Si existe stockout,
        # suponemos que la empresa puede realizar compras urgentes.
        #
        # ==========================================================================================

        emergency_cost = (

            stockout_units

            *
            EMERGENCY_COST_PER_UNIT

        )


        # ==========================================================================================
        # STEP 16. SUPPLY RISK COST
        # ==========================================================================================
        #
        # Agregamos un costo relacionado con:
        #
        # baja confiabilidad
        # y
        # lead times largos.
        #
        #
        # Es un costo pedagógico para representar exposición.
        #
        # ==========================================================================================

        risk_cost = (

            (1 - reliability)

            *
            lead_time

            *
            1500

        )


        # ==========================================================================================
        # STEP 17. TOTAL COST
        # ==========================================================================================

        total_cost = (

            inventory_holding_cost

            +

            stockout_cost

            +

            emergency_cost

            +

            risk_cost

            +

            parameters["strategy_fixed_cost"]

        )


        # ==========================================================================================
        # STEP 18. SAVE RESULTS
        # ==========================================================================================

        results.append({

            "Strategy": strategy_name,

            "Simulation": simulation + 1,

            "Demand": demand,

            "Lead_Time": lead_time,

            "Reliability": reliability,

            "Supply_Received": actual_supply,

            "Adjusted_Demand": adjusted_demand,

            "Ending_Inventory": ending_inventory,

            "Stockout": stockout,

            "Stockout_Units": stockout_units,

            "Service_Level": service_level,

            "Inventory_Holding_Cost": inventory_holding_cost,

            "Stockout_Cost": stockout_cost,

            "Emergency_Cost": emergency_cost,

            "Risk_Cost": risk_cost,

            "Total_Cost": total_cost

        })


    # Convertimos resultados en DataFrame.
    return pd.DataFrame(results)


# ==================================================================================================
# 7. RUN MONTE CARLO FOR ALL STRATEGIES
# ==================================================================================================

all_results = []


for strategy_name, parameters in strategies.items():

    simulation_result = monte_carlo_strategy(

        strategy_name,

        parameters

    )

    all_results.append(
        simulation_result
    )


# Unimos todas las simulaciones.
results = pd.concat(

    all_results,

    ignore_index=True

)


# ==================================================================================================
# 8. CHECK THE DATA
# ==================================================================================================

print("\n" + "=" * 120)

print("FIRST MONTE CARLO RESULTS")

print("=" * 120)


print(

    results.head(10).round(2)

)


# ==================================================================================================
# 9. SUMMARY OF MONTE CARLO RESULTS
# ==================================================================================================
#
# Ahora vamos a transformar miles de simulaciones
# en indicadores gerenciales.
#
# ==================================================================================================


summary = (

    results
    .groupby("Strategy")
    .agg(

        Average_Demand=("Demand", "mean"),

        Average_Lead_Time=("Lead_Time", "mean"),

        Average_Reliability=("Reliability", "mean"),

        Average_Inventory=("Ending_Inventory", "mean"),

        Stockout_Probability=("Stockout", "mean"),

        Average_Service_Level=("Service_Level", "mean"),

        Average_Total_Cost=("Total_Cost", "mean"),

        Cost_Std_Deviation=("Total_Cost", "std"),

        Maximum_Cost=("Total_Cost", "max")

    )

    .reset_index()

)


# Convertimos Stockout Probability a porcentaje.

summary["Stockout_Probability"] = (

    summary["Stockout_Probability"]

    * 100

)


print("\n" + "=" * 120)

print("MONTE CARLO MANAGEMENT SUMMARY")

print("=" * 120)


print(

    summary.round(2).to_string(index=False)

)


# ==================================================================================================
# 10. INTERPRETING THE FIRST RESULT
# ==================================================================================================
#
# Aquí ya tenemos algo muy importante:
#
#
# Average Total Cost
#       NO muestra todo el riesgo.
#
#
# Dos estrategias pueden tener:
#
#         Average Cost ≈ similar
#
# pero:
#
#         Standard Deviation ≠ similar
#
#
# Esto significa que una puede ser:
#
# MÁS PREDECIBLE
#
# y otra:
#
# MÁS VOLÁTIL.
#
# ==================================================================================================


# ==================================================================================================
# 11. COLOR PALETTE
# ==================================================================================================

strategy_colors = {

    "Current Policy": "#E74C3C",

    "Higher Safety Stock": "#F39C12",

    "Dual Sourcing": "#3498DB",

    "Nearshoring": "#2ECC71"

}


# ==================================================================================================
# 12. GRAPH 1
#
# DISTRIBUTION OF TOTAL COST
# ==================================================================================================
#
# Un histograma permite observar:
#
# - dónde se concentran los costos
# - qué tan dispersos son
# - si existen escenarios extremos
#
# ==================================================================================================


plt.figure(
    figsize=(14, 8)
)


for strategy in strategies.keys():

    subset = results[
        results["Strategy"] == strategy
    ]

    plt.hist(

        subset["Total_Cost"],

        bins=35,

        alpha=0.45,

        label=strategy,

        color=strategy_colors[strategy]

    )


plt.title(

    "Monte Carlo Distribution of Total Supply Chain Cost",

    fontsize=18,

    fontweight="bold"

)


plt.xlabel(
    "Total Cost"
)


plt.ylabel(
    "Frequency"
)


plt.legend()


plt.grid(
    alpha=0.20
)


plt.tight_layout()

plt.show()


# ==================================================================================================
# INTERPRETATION
# ==================================================================================================
#
# IMPORTANTE PARA LOS ESTUDIANTES:
#
# No debemos observar solamente el promedio.
#
# Una distribución:
#
# estrecha  = resultados más predecibles
#
# ancha     = mayor variabilidad / incertidumbre
#
# cola larga hacia la derecha
#           = posibilidad de costos extremos
#
# ==================================================================================================


# ==================================================================================================
# 13. GRAPH 2
#
# BOX PLOT OF TOTAL COST
# ==================================================================================================
#
# El boxplot permite observar:
#
# - mediana
# - dispersión
# - rango
# - valores extremos
#
# ==================================================================================================


data_boxplot = [

    results[
        results["Strategy"] == strategy
    ]["Total_Cost"]

    for strategy in strategies.keys()

]


plt.figure(
    figsize=(13, 7)
)


box = plt.boxplot(

    data_boxplot,

    labels=list(strategies.keys()),

    patch_artist=True

)


for patch, strategy in zip(

    box["boxes"],

    strategies.keys()

):

    patch.set_facecolor(
        strategy_colors[strategy]
    )


plt.title(

    "Total Cost Variability by Strategy",

    fontsize=18,

    fontweight="bold"

)


plt.ylabel(
    "Total Cost"
)


plt.xticks(
    rotation=15
)


plt.grid(
    axis="y",
    alpha=0.20
)


plt.tight_layout()

plt.show()


# ==================================================================================================
# 14. GRAPH 3
#
# STOCKOUT PROBABILITY
# ==================================================================================================


stockout_summary = (

    results
    .groupby("Strategy")["Stockout"]
    .mean()

    * 100

)


plt.figure(
    figsize=(12, 7)
)


bars = plt.bar(

    stockout_summary.index,

    stockout_summary.values,

    color=[
        strategy_colors[s]
        for s in stockout_summary.index
    ]

)


plt.title(

    "Probability of Stockout",

    fontsize=18,

    fontweight="bold"

)


plt.ylabel(
    "Stockout Probability (%)"
)


for bar in bars:

    height = bar.get_height()

    plt.text(

        bar.get_x()
        +
        bar.get_width() / 2,

        height + 0.5,

        f"{height:.1f}%",

        ha="center",

        fontweight="bold"

    )


plt.xticks(
    rotation=15
)


plt.grid(
    axis="y",
    alpha=0.20
)


plt.tight_layout()

plt.show()


# ==================================================================================================
# KEY QUESTION
# ==================================================================================================
#
# ¿Qué estrategia tiene menor probabilidad
# de experimentar un stockout?
#
#
# Pero también:
#
# ¿Cuánto debemos pagar para reducir esa probabilidad?
#
# ==================================================================================================


# ==================================================================================================
# 15. GRAPH 4
#
# AVERAGE SERVICE LEVEL
# ==================================================================================================


service_summary = (

    results
    .groupby("Strategy")["Service_Level"]
    .mean()

)


plt.figure(
    figsize=(12, 7)
)


bars = plt.bar(

    service_summary.index,

    service_summary.values,

    color=[
        strategy_colors[s]
        for s in service_summary.index
    ]

)


plt.title(

    "Average Service Level Under Uncertainty",

    fontsize=18,

    fontweight="bold"

)


plt.ylabel(
    "Service Level (%)"
)


plt.ylim(
    85,
    101
)


# Objetivo de ejemplo:
# Service Level >= 96%

plt.axhline(

    96,

    color="#34495E",

    linestyle="--",

    linewidth=2,

    label="Service Target = 96%"

)


for bar in bars:

    height = bar.get_height()

    plt.text(

        bar.get_x()
        +
        bar.get_width()/2,

        height + 0.2,

        f"{height:.1f}%",

        ha="center",

        fontweight="bold"

    )


plt.legend()

plt.xticks(
    rotation=15
)

plt.grid(
    axis="y",
    alpha=0.20
)

plt.tight_layout()

plt.show()


# ==================================================================================================
# 16. GRAPH 5
#
# SERVICE LEVEL DISTRIBUTION
# ==================================================================================================
#
# Dos estrategias pueden presentar:
#
# promedio parecido
#
# pero:
#
# una puede ser mucho más estable.
#
# ==================================================================================================


plt.figure(
    figsize=(14, 8)
)


for strategy in strategies.keys():

    subset = results[
        results["Strategy"] == strategy
    ]

    plt.hist(

        subset["Service_Level"],

        bins=30,

        alpha=0.40,

        label=strategy,

        color=strategy_colors[strategy]

    )


plt.title(

    "Distribution of Service Level",

    fontsize=18,

    fontweight="bold"

)


plt.xlabel(
    "Service Level (%)"
)


plt.ylabel(
    "Frequency"
)


plt.legend()

plt.grid(
    alpha=0.20
)

plt.tight_layout()

plt.show()


# ==================================================================================================
# 17. GRAPH 6
#
# COST VS SERVICE
# ==================================================================================================
#
# Cada punto representa una estrategia.
#
#
# X = Average Cost
#
# Y = Average Service
#
# Bubble Size = Probability of Stockout
#
#
# Idealmente queremos:
#
# BAJO COSTO
# ALTO SERVICIO
# BURBUJA PEQUEÑA
#
# ==================================================================================================


comparison = summary.copy()


plt.figure(
    figsize=(13, 8)
)


for _, row in comparison.iterrows():

    plt.scatter(

        row["Average_Total_Cost"],

        row["Average_Service_Level"],

        s=(
            row["Stockout_Probability"]
            + 1
        ) * 60,

        color=strategy_colors[
            row["Strategy"]
        ],

        alpha=0.75,

        edgecolor="black"

    )


    plt.text(

        row["Average_Total_Cost"] + 1000,

        row["Average_Service_Level"] + 0.10,

        row["Strategy"],

        fontsize=10

    )


plt.title(

    "Cost – Service – Stockout Risk Trade-Off",

    fontsize=18,

    fontweight="bold"

)


plt.xlabel(
    "Average Total Cost"
)


plt.ylabel(
    "Average Service Level (%)"
)


plt.grid(
    alpha=0.20
)


plt.tight_layout()

plt.show()


# ==================================================================================================
# 18. IDENTIFY EXTREME COST SCENARIOS
# ==================================================================================================
#
# Hasta ahora analizamos principalmente promedios.
#
# Pero Monte Carlo también permite estudiar:
#
# EXTREME EVENTS.
#
#
# Vamos a calcular:
#
# P90 = Percentile 90
#
# P95 = Percentile 95
#
# P99 = Percentile 99
#
#
# P95 significa:
#
# 95% de las simulaciones tienen un costo
# inferior a ese valor.
#
# Solo 5% presentan costos superiores.
#
# ==================================================================================================


risk_percentiles = (

    results
    .groupby("Strategy")["Total_Cost"]
    .agg(

        Mean="mean",

        Median="median",

        P90=lambda x: np.percentile(x, 90),

        P95=lambda x: np.percentile(x, 95),

        P99=lambda x: np.percentile(x, 99)

    )

    .reset_index()

)


print("\n" + "=" * 120)

print("COST RISK PERCENTILES")

print("=" * 120)


print(

    risk_percentiles
    .round(2)
    .to_string(index=False)

)


# ==================================================================================================
# 19. GRAPH 7
#
# AVERAGE COST VS P95 COST
# ==================================================================================================
#
# Esta gráfica permite explicar algo esencial:
#
#
# EXPECTED COST
#
# no es igual a
#
# RISK EXPOSURE.
#
#
# Una estrategia puede tener un costo promedio atractivo
# pero un P95 muy elevado.
#
# ==================================================================================================


x = np.arange(
    len(risk_percentiles)
)


width = 0.35


plt.figure(
    figsize=(13, 7)
)


plt.bar(

    x - width/2,

    risk_percentiles["Mean"],

    width,

    label="Average Cost",

    color="#3498DB"

)


plt.bar(

    x + width/2,

    risk_percentiles["P95"],

    width,

    label="P95 Cost",

    color="#E74C3C"

)


plt.xticks(

    x,

    risk_percentiles["Strategy"],

    rotation=15

)


plt.ylabel(
    "Cost"
)


plt.title(

    "Expected Cost vs Extreme Cost Exposure",

    fontsize=18,

    fontweight="bold"

)


plt.legend()

plt.grid(
    axis="y",
    alpha=0.20
)

plt.tight_layout()

plt.show()


# ==================================================================================================
# 20. VALUE AT RISK STYLE MEASURE
# ==================================================================================================
#
# Podemos crear una medida simple de "Cost at Risk".
#
#
# Cost-at-Risk = P95 - Average Cost
#
#
# Cuanto mayor sea:
#
# mayor es la diferencia entre
# comportamiento normal y escenario adverso.
#
# ==================================================================================================


risk_percentiles["Cost_at_Risk"] = (

    risk_percentiles["P95"]

    -

    risk_percentiles["Mean"]

)


print("\nCOST AT RISK")

print(

    risk_percentiles[
        [
            "Strategy",
            "Mean",
            "P95",
            "Cost_at_Risk"
        ]
    ]
    .round(2)
    .to_string(index=False)

)


# ==================================================================================================
# 21. IDENTIFY CRITICAL SCENARIOS
# ==================================================================================================
#
# Vamos a definir un escenario crítico como aquel que presenta:
#
# Service Level < 95%
#
# O
#
# Stockout > 0
#
# O
#
# Total Cost > Percentil 95 global
#
# ==================================================================================================


global_p95_cost = np.percentile(

    results["Total_Cost"],

    95

)


results["Critical_Scenario"] = (

    (
        results["Service_Level"] < 95
    )

    |

    (
        results["Stockout"] == 1
    )

    |

    (
        results["Total_Cost"] > global_p95_cost
    )

)


critical_probability = (

    results
    .groupby("Strategy")["Critical_Scenario"]
    .mean()

    * 100

)


print("\n" + "=" * 120)

print("PROBABILITY OF A CRITICAL SCENARIO")

print("=" * 120)


print(

    critical_probability
    .round(2)

)


# ==================================================================================================
# 22. GRAPH 8
#
# PROBABILITY OF CRITICAL SCENARIOS
# ==================================================================================================


plt.figure(
    figsize=(12, 7)
)


bars = plt.bar(

    critical_probability.index,

    critical_probability.values,

    color=[
        strategy_colors[s]
        for s in critical_probability.index
    ]

)


plt.title(

    "Probability of Critical Supply Chain Scenarios",

    fontsize=18,

    fontweight="bold"

)


plt.ylabel(
    "Probability (%)"
)


for bar in bars:

    height = bar.get_height()

    plt.text(

        bar.get_x()
        +
        bar.get_width()/2,

        height + 0.5,

        f"{height:.1f}%",

        ha="center",

        fontweight="bold"

    )


plt.xticks(
    rotation=15
)

plt.grid(
    axis="y",
    alpha=0.20
)

plt.tight_layout()

plt.show()


# ==================================================================================================
# 23. WHAT CONDITIONS CREATE THE WORST RESULTS?
# ==================================================================================================
#
# Esta sección es particularmente interesante.
#
# Queremos analizar:
#
# ¿Qué combinación de variables genera
# los peores resultados?
#
#
# Seleccionaremos el 5% de escenarios
# con mayor costo.
#
# ==================================================================================================


cost_threshold = np.percentile(

    results["Total_Cost"],

    95

)


worst_cases = results[

    results["Total_Cost"]
    >=
    cost_threshold

]


print("\n" + "=" * 120)

print("CHARACTERISTICS OF THE WORST 5% OF SCENARIOS")

print("=" * 120)


print(

    worst_cases[
        [
            "Demand",
            "Lead_Time",
            "Reliability",
            "Stockout_Units",
            "Service_Level",
            "Total_Cost"
        ]
    ]

    .mean()

    .round(2)

)


# ==================================================================================================
# INTERPRETATION
# ==================================================================================================
#
# Si los peores escenarios muestran:
#
# HIGH DEMAND
# +
# LONG LEAD TIME
# +
# LOW RELIABILITY
#
#
# podemos interpretar que el verdadero problema
# no es una variable aislada.
#
#
# El riesgo surge de la combinación de incertidumbres.
#
# ==================================================================================================


# ==================================================================================================
# 24. GRAPH 9
#
# DEMAND VS LEAD TIME
#
# COLOR = SERVICE LEVEL
# SIZE = TOTAL COST
# ==================================================================================================
#
# Esta gráfica permite visualizar
# dónde aparecen las situaciones críticas.
#
# ==================================================================================================


sample_results = results.sample(

    1000,

    random_state=42

)


plt.figure(
    figsize=(13, 8)
)


scatter = plt.scatter(

    sample_results["Lead_Time"],

    sample_results["Demand"],

    c=sample_results["Service_Level"],

    s=(
        sample_results["Total_Cost"]
        /
        sample_results["Total_Cost"].max()
        *
        400
        +
        20
    ),

    cmap="RdYlGn",

    alpha=0.65,

    edgecolor="black",

    linewidth=0.3

)


plt.colorbar(
    scatter,
    label="Service Level (%)"
)


plt.xlabel(
    "Supplier Lead Time (Days)"
)


plt.ylabel(
    "Demand"
)


plt.title(

    "Supply Chain Risk Map: Demand vs Lead Time",

    fontsize=18,

    fontweight="bold"

)


plt.grid(
    alpha=0.15
)


plt.tight_layout()

plt.show()


# ==================================================================================================
# 25. GRAPH 10
#
# RELIABILITY VS TOTAL COST
# ==================================================================================================


plt.figure(
    figsize=(13, 8)
)


for strategy in strategies.keys():

    subset = results[
        results["Strategy"] == strategy
    ]


    plt.scatter(

        subset["Reliability"],

        subset["Total_Cost"],

        alpha=0.30,

        s=20,

        color=strategy_colors[strategy],

        label=strategy

    )


plt.title(

    "Supplier Reliability vs Total Supply Chain Cost",

    fontsize=18,

    fontweight="bold"

)


plt.xlabel(
    "Supplier Reliability"
)


plt.ylabel(
    "Total Cost"
)


plt.legend()

plt.grid(
    alpha=0.20
)

plt.tight_layout()

plt.show()


# ==================================================================================================
# INTERPRETATION
# ==================================================================================================
#
# Esta gráfica ayuda a analizar:
#
# ¿Qué ocurre cuando disminuye la confiabilidad?
#
#
# Esperaríamos normalmente:
#
# reliability ↓
#
# stockout risk ↑
#
# emergency cost ↑
#
# total cost ↑
#
# ==================================================================================================


# ==================================================================================================
# 26. CREATE A MANAGEMENT SCORE
# ==================================================================================================
#
# Ahora transformaremos los resultados
# en una herramienta para decisión.
#
#
# No es una fórmula universal.
#
# Es una herramienta pedagógica.
#
#
# Evaluaremos:
#
# COST        35%
# SERVICE     30%
# RISK        25%
# STABILITY   10%
#
# ==================================================================================================


decision = summary.copy()


# --------------------------------------------------------------------------------------------------
# COST SCORE
#
# Menor costo = mejor.
# --------------------------------------------------------------------------------------------------

decision["Cost_Score"] = (

    100

    *

    (
        decision["Average_Total_Cost"].max()

        -

        decision["Average_Total_Cost"]
    )

    /

    (
        decision["Average_Total_Cost"].max()

        -

        decision["Average_Total_Cost"].min()
    )

)


# --------------------------------------------------------------------------------------------------
# SERVICE SCORE
#
# Mayor service level = mejor.
# --------------------------------------------------------------------------------------------------

decision["Service_Score"] = (

    100

    *

    (
        decision["Average_Service_Level"]

        -

        decision["Average_Service_Level"].min()
    )

    /

    (
        decision["Average_Service_Level"].max()

        -

        decision["Average_Service_Level"].min()
    )

)


# --------------------------------------------------------------------------------------------------
# RISK SCORE
#
# Menor probabilidad de stockout = mejor.
# --------------------------------------------------------------------------------------------------

decision["Risk_Score"] = (

    100

    *

    (
        decision["Stockout_Probability"].max()

        -

        decision["Stockout_Probability"]
    )

    /

    (
        decision["Stockout_Probability"].max()

        -

        decision["Stockout_Probability"].min()
    )

)


# --------------------------------------------------------------------------------------------------
# STABILITY SCORE
#
# Menor desviación del costo = más estabilidad.
# --------------------------------------------------------------------------------------------------

decision["Stability_Score"] = (

    100

    *

    (
        decision["Cost_Std_Deviation"].max()

        -

        decision["Cost_Std_Deviation"]
    )

    /

    (
        decision["Cost_Std_Deviation"].max()

        -

        decision["Cost_Std_Deviation"].min()
    )

)


# ==================================================================================================
# 27. FINAL BALANCE SCORE
# ==================================================================================================


decision["Overall_Score"] = (

    decision["Cost_Score"] * 0.35

    +

    decision["Service_Score"] * 0.30

    +

    decision["Risk_Score"] * 0.25

    +

    decision["Stability_Score"] * 0.10

)


decision = decision.sort_values(

    "Overall_Score",

    ascending=False

)


print("\n" + "=" * 120)

print("FINAL MANAGEMENT RANKING")

print("=" * 120)


print(

    decision[
        [
            "Strategy",
            "Average_Total_Cost",
            "Average_Service_Level",
            "Stockout_Probability",
            "Cost_Std_Deviation",
            "Overall_Score"
        ]
    ]

    .round(2)

    .to_string(index=False)

)


# ==================================================================================================
# 28. GRAPH 11
#
# FINAL STRATEGY RANKING
# ==================================================================================================


ranking_plot = decision.sort_values(

    "Overall_Score"

)


plt.figure(
    figsize=(11, 7)
)


bars = plt.barh(

    ranking_plot["Strategy"],

    ranking_plot["Overall_Score"],

    color=[
        strategy_colors[s]
        for s in ranking_plot["Strategy"]
    ]

)


plt.title(

    "Overall Cost – Service – Risk Strategy Ranking",

    fontsize=18,

    fontweight="bold"

)


plt.xlabel(
    "Overall Decision Score"
)


for bar in bars:

    width = bar.get_width()

    plt.text(

        width + 1,

        bar.get_y()
        +
        bar.get_height()/2,

        f"{width:.1f}",

        va="center",

        fontweight="bold"

    )


plt.grid(
    axis="x",
    alpha=0.20
)


plt.tight_layout()

plt.show()


# ==================================================================================================
# 29. RADAR CHART
# ==================================================================================================
#
# El radar permitirá observar simultáneamente:
#
# COST
# SERVICE
# RISK
# STABILITY
#
#
# Cuanto más hacia afuera:
#
# mejor desempeño.
#
# ==================================================================================================


radar_variables = [

    "Cost_Score",

    "Service_Score",

    "Risk_Score",

    "Stability_Score"

]


radar_labels = [

    "Cost Efficiency",

    "Service",

    "Risk Protection",

    "Stability"

]


N = len(
    radar_variables
)


angles = np.linspace(

    0,

    2 * np.pi,

    N,

    endpoint=False

).tolist()


angles += angles[:1]


fig = plt.figure(
    figsize=(10, 10)
)


ax = plt.subplot(
    111,
    polar=True
)


for _, row in decision.iterrows():

    values = [

        row[column]

        for column in radar_variables

    ]


    values += values[:1]


    ax.plot(

        angles,

        values,

        linewidth=2.3,

        label=row["Strategy"],

        color=strategy_colors[
            row["Strategy"]
        ]

    )


    ax.fill(

        angles,

        values,

        alpha=0.05,

        color=strategy_colors[
            row["Strategy"]
        ]

    )


ax.set_xticks(
    angles[:-1]
)


ax.set_xticklabels(

    radar_labels,

    fontsize=11

)


ax.set_title(

    "Monte Carlo Strategy Performance Profile",

    fontsize=18,

    fontweight="bold",

    pad=30

)


ax.legend(

    bbox_to_anchor=(1.35, 1.10)

)


plt.show()


# ==================================================================================================
# 30. AUTOMATIC MANAGEMENT INTERPRETATION
# ==================================================================================================


best_strategy = decision.iloc[0]


print("\n" + "=" * 120)

print("MANAGEMENT RECOMMENDATION")

print("=" * 120)


print(
    f"""

According to the assumptions used in this Monte Carlo model,
the strategy with the best overall balance is:

        {best_strategy['Strategy']}

Average Total Cost:
        {best_strategy['Average_Total_Cost']:,.2f}

Average Service Level:
        {best_strategy['Average_Service_Level']:.2f}%

Stockout Probability:
        {best_strategy['Stockout_Probability']:.2f}%

Overall Decision Score:
        {best_strategy['Overall_Score']:.2f}


IMPORTANT:

This strategy is not universally optimal.

The result depends on:

- Demand uncertainty
- Lead-time variability
- Supplier reliability
- Cost assumptions
- Service targets
- Risk tolerance
- Product criticality
- Management priorities

Changing these assumptions may change the ranking.

"""
)


# ==================================================================================================
# 31. FINAL QUESTIONS FOR STUDENTS
# ==================================================================================================

print("\n" + "=" * 120)

print("MANAGEMENT DISCUSSION")

print("=" * 120)


print(
"""

1. Which strategy has the lowest expected total cost?

2. Which strategy provides the highest average service level?

3. Which strategy has the lowest probability of stockout?

4. Which strategy presents the lowest cost variability?

5. Which strategy has the lowest P95 cost?

6. Is the strategy with the lowest average cost also the safest?

7. Which variables appear most frequently in the worst scenarios?

8. What happens when high demand, long lead time,
   and low supplier reliability occur simultaneously?

9. Does increasing safety stock eliminate supply chain risk?

10. When does dual sourcing create enough risk reduction
    to justify its additional cost?

11. Would nearshoring still be attractive if its fixed cost
    increased significantly?

12. Which strategy would you recommend for a highly critical product?

13. Which strategy would you recommend for a low-value,
    stable-demand product?

14. How would your decision change if management placed
    twice as much importance on service level?

15. What evidence from the Monte Carlo simulation would you use
    to defend your recommendation to senior management?

"""
)